# Getting started

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LukasLiss/totem-tool/blob/main/totem_lib/docs/examples/getting_started.ipynb)

This notebook loads an object-centric event log, discovers a TOTeM model from it, and groups the object types into process areas. It runs in under a minute.

## Setup

In Google Colab, the first cell installs totem-lib. Everywhere else it does nothing, so install totem-lib first as shown in [Installation](../installation.md).

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install --quiet "git+https://github.com/LukasLiss/totem-tool.git#subdirectory=totem_lib"

The example log comes from the totem-lib repository. Outside a clone of the repository, the next cell downloads it (16 MB).

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

LOG = Path("../../example_data/ContainerLogistics.sqlite")
if not LOG.exists():
    LOG = Path("ContainerLogistics.sqlite")
    if not LOG.exists():
        urlretrieve(
            "https://raw.githubusercontent.com/LukasLiss/totem-tool/main/"
            "totem_lib/example_data/ContainerLogistics.sqlite",
            LOG,
        )

## Load an event log

`import_ocel` reads OCEL 2.0 files. It picks the format from the file extension: `.sqlite`, `.json`, `.xml`, `.csv` or `.duckdb`.

In [ ]:
from totem_lib import import_ocel

ocel = import_ocel(LOG)
print(f"{ocel.events.height} events and {ocel.objects.height} objects")
sorted(ocel.object_types)

The events and objects are [Polars](https://pola.rs) DataFrames:

In [ ]:
ocel.events.head()

## Discover a TOTeM model

A TOTeM (Temporal Object Type Model) describes how each pair of object types relates. With `tau=0.9`, a relation is only reported if it holds for at least 90% of the observations.

In [ ]:
from totem_lib import totemDiscovery

# totemDiscovery prints progress messages while it runs.
totem = totemDiscovery(ocel, tau=0.9)

`totem_to_dict` turns the model into plain Python data. The **cardinalities** say how many objects of type `to` go with one object of type `from`:

- `log_cardinality`: over the whole log
- `event_cardinality`: within a single event

In [ ]:
import polars as pl
from totem_lib import totem_to_dict

pl.Config.set_tbl_rows(20)  # show every row of these small tables
model = totem_to_dict(totem)
pl.DataFrame(model["cardinalities"])

The **temporal relations** compare the lifespans of the objects, from their first to their last event:

- `D` (during): the `from` object lives within the lifespan of the `to` object
- `I` (initiating): the `from` object starts first
- `P` (parallel): neither of these holds reliably

In [ ]:
pl.DataFrame(
    [
        {"from": source, "to": target, "relation": relation}
        for relation, pairs in model["tempgraph"].items()
        if relation != "nodes"
        for source, target in pairs
    ]
)

## Find process areas

`mlpaDiscovery` uses the TOTeM model to put every object type on a layer. Connected types on the same layer form a process area, together with the activities that belong to it.

In [ ]:
from totem_lib import mlpaDiscovery

# mlpaDiscovery prints the solver log while it runs.
process_view = mlpaDiscovery(totem)

In [ ]:
for layer, areas in sorted(process_view.items()):
    for object_types, activities in areas:
        print(f"Layer {int(layer)}: {', '.join(object_types)}")
        print(f"  activities: {', '.join(sorted(activities)) or 'none'}")

## Next steps

The [API reference](../api/index.rst) lists every public function and class, including the other models: directly-follows graphs, Petri nets and causal nets.